# Text classification

Using the dataset `dataset_emails.csv` create three text classificators:
- Using rule-based approach (regex)
- Using naive-bayes
- Using Spacy 3 

Compare the results and show what is better and why. 

In [1]:
import pandas as pd
import re

from sklearn.metrics import accuracy_score, classification_report

## Rule-based approach

### Data preprocessing

In [2]:
df = pd.read_csv(
    'data/dataset_emails.csv',
    on_bad_lines='skip'
)

In [3]:
def text_preprocessing(text):
    """lowercase, no blank spaces and no special characters"""

    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text

df['prompt_procesado'] = df['prompt'].apply(text_preprocessing)

### Regex rules

This approach is based on finding commond words that we may consider to appear in each group

In [4]:
rules = {
    'send': [
        r'\bsend\b',
        r'\bcompose\b',
        r'\benviar\b',
        r'\bredactar\b'
    ],
    'list trash': [
        r'\blist trash\b',
        r'\btrash list\b',
        r'\blist deleted\b'
    ],
    'read': [
        r'\bread\b',
        r'\bleer\b'
    ],
    'reply': [
        r'\breply\b',
        r'\brespond\b',
        r'\bresponder\b'
    ],
    'untrash': [
        r'\buntrash\b',
        r'\brestore\b',
        r'\brestore\b',
        r'\brecuperar\b'
    ],
    'forward': [
        r'\bforward\b',
        r'\breenviar\b'
    ],
    'star': [
        r'\bstar\b',
        r'\bfavorite\b',
        r'\bmark as important\b'
    ],
    'trash_list': [
        r'\btrash list\b',
        r'\blist trash\b'
    ]
}

### Prediction

In [5]:
def classify_prompt(texto):
    for category, pattern in rules.items():
        for patron in pattern:
            if re.search(patron, texto):
                return category
    return "unkwom"

In [6]:
df['prediccion'] = df['prompt_procesado'].apply(classify_prompt)

display(df.head())


,prompt,label,prompt_procesado,prediccion
0,"Can I send an email, please?",send,can i send an email please,send
1,I'd like to compose an email.,send,i d like to compose an email,send
2,I need to send an email.,send,i need to send an email,send
3,Could you help me write an email?,send,could you help me write an email,unkwom
4,Is it possible to send an email with you?,send,is it possible to send an email with you,send


### Metrics

In [7]:
exactitud = accuracy_score(df['label'], df['prediccion'])
print(f"Accuracy: {exactitud:.4f}")

Accuracy: 0.2381


In [8]:
# Precision, recall, f1-score
report = classification_report(
    df['label'],
    # vs
    df['prediccion']
)
print(f"{'*'*24} Report {'*'*24}\n {report}")
print(f"{'*'*57}")

************************ Report ************************
               precision    recall  f1-score   support

     forward       1.00      0.39      0.56       100
        list       0.00      0.00      0.00       100
        read       0.94      0.29      0.44       100
       reply       0.99      0.73      0.84       100
        send       0.52      0.34      0.41       100
        star       0.98      0.47      0.64       100
       trash       0.00      0.00      0.00       100
  trash_list       0.00      0.00      0.00       100
     unknown       0.00      0.00      0.00       100
      unkwom       0.00      0.00      0.00         0
     untrash       1.00      0.15      0.27        91

    accuracy                           0.24       991
   macro avg       0.49      0.22      0.29       991
weighted avg       0.54      0.24      0.32       991

*********************************************************


/home/santi/anaconda3/envs/nlp/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/santi/anaconda3/envs/nlp/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/santi/anaconda3/envs/nlp/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result

## Spacy 3 approach 

⚠️ Warning: remember to change the kernel from `nlp` to `spacy` as in the first one spacy does not work.

In [20]:
import spacy
from spacy.matcher import Matcher
import pandas as pd

### Load english language and instace the matcher

In [21]:
nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

### Generate rules for matcher

In [24]:
patterns = {
    "send": [
        [{"LOWER": "send"}],
        [{"LOWER": "compose"}],
        [{"LOWER": "enviar"}],
        [{"LOWER": "redactar"}]
    ],
    "list trash": [
        [{"LOWER": "list"}, {"LOWER": "trash"}],
        [{"LOWER": "trash"}, {"LOWER": "list"}],
        [{"LOWER": "list"}, {"LOWER": "deleted"}]
    ],
    "read": [
        [{"LOWER": "read"}],
        [{"LOWER": "leer"}]
    ],
    "reply": [
        [{"LOWER": "reply"}],
        [{"LOWER": "respond"}],
    ],
    "untrash": [
        [{"LOWER": "untrash"}],
        [{"LOWER": "restore"}],
    ],
    "forward": [
        [{"LOWER": "forward"}],
    ],
    "star": [
        [{"LOWER": "star"}],
        [{"LOWER": "favourite"}],
        [{"LOWER": "mark"}, {"LOWER": "as"}, {"LOWER": "important"}]
    ],
    "trash_list": [
        [{"LOWER": "trash"}, {"LOWER": "list"}],
        [{"LOWER": "list"}, {"LOWER": "trash"}]
    ]
}

### Classify promts

In [25]:
for label, pattern_list in patterns.items():
    matcher.add(label, pattern_list)

def classify_prompt_spacy(text):
    """Classify a prompt using Spacy pattern matching"""
    doc = nlp(text.strip())
    matches = matcher(doc)
    if matches:
        # Take the first match found
        match_id, start, end = matches[0]
        label = nlp.vocab.strings[match_id]
        return label
    return "unkwom"

In [ ]:
df = pd.read_csv('data/dataset_emails.csv', on_bad_lines='skip')

# Aplicamos la función de clasificación a la columna 'prompt'
df['prediccion'] = df['prompt'].apply(clasificar_prompt_spacy)

# Mostramos los resultados: prompt original, etiqueta real y predicción
display(df.head())


,prompt,label,prediccion
0,"Can I send an email, please?",send,send
1,I'd like to compose an email.,send,send
2,I need to send an email.,send,send
3,Could you help me write an email?,send,unkwom
4,Is it possible to send an email with you?,send,send


### Metrics

In [8]:
exactitud_spacy = accuracy_score(df['label'], df['prediccion'])
print(f"Spacy Model Accuracy: {exactitud_spacy:.4f}")

report_spacy = classification_report(df['label'], df['prediccion'])
print(f"{'*'*24} Spacy Model Classification Report {'*'*24}\n{report_spacy}")

Spacy Model Accuracy: 0.2381
************************ Spacy Model Classification Report ************************
              precision    recall  f1-score   support

     forward       1.00      0.39      0.56       100
        list       0.00      0.00      0.00       100
        read       0.94      0.29      0.44       100
       reply       0.99      0.73      0.84       100
        send       0.52      0.34      0.41       100
        star       0.98      0.47      0.64       100
       trash       0.00      0.00      0.00       100
  trash_list       0.00      0.00      0.00       100
     unknown       0.00      0.00      0.00       100
      unkwom       0.00      0.00      0.00         0
     untrash       1.00      0.15      0.27        91

    accuracy                           0.24       991
   macro avg       0.49      0.22      0.29       991
weighted avg       0.54      0.24      0.32       991



/home/santi/anaconda3/envs/spacy/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/santi/anaconda3/envs/spacy/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/santi/anaconda3/envs/spacy/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

## Using naive-bayes approach


⚠️ Warning: remember to change the kernel from `spacy` to `nlp`

In [9]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [10]:
df = pd.read_csv('data/dataset_emails.csv', on_bad_lines='skip')

def procces_texto(texto):
    """lowercase, no blank spaces and no special characters"""
    texto = texto.lower().strip()
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return texto

df['prompt_procesado'] = df['prompt'].apply(procces_texto)

In [11]:
#random = 33

X_train, X_test, y_train, y_test = train_test_split(
    df['prompt_procesado'], df['label'], 
    test_size=0.2, 
    #random_state=random
)

### Vectorization and NB model

In [12]:
# Vectorization
vectorizer = TfidfVectorizer()
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# Train Naive Bayes
clf = MultinomialNB()
clf.fit(X_train_vect, y_train)

# Predecir sobre el conjunto de prueba
y_pred = clf.predict(X_test_vect)

### Metrics

In [13]:
exactitud = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy): {exactitud:.4f}")

Exactitud (Accuracy): 0.8050


In [14]:
# precisión, recall, F1
print("Reporte de Clasificación:\n")
print(classification_report(y_test, y_pred))

Reporte de Clasificación:

              precision    recall  f1-score   support

     forward       0.76      0.96      0.85        23
        list       0.93      0.62      0.74        21
        read       0.86      0.63      0.73        19
       reply       0.74      0.95      0.83        21
        send       0.67      0.59      0.62        17
        star       0.89      0.96      0.93        26
       trash       0.94      0.81      0.87        21
  trash_list       0.68      0.93      0.79        14
     unknown       1.00      0.65      0.79        20
     untrash       0.70      0.89      0.78        18

    accuracy                           0.81       200
   macro avg       0.82      0.80      0.79       200
weighted avg       0.83      0.81      0.80       200



## Report